# Lab 21-01 · S-Parameters, Calibration and Stability

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import cmath
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import sparam as sp

P = lambda m, a: m * cmath.exp(1j * math.radians(a))
rng = np.random.default_rng(21)


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {m.group(1): float(m.group(2)) for m in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Reflections

*Predict* which loads give the same return loss.

In [2]:
for z in (10, 25, 50, 75, 100, 250):
    g = sp.gamma(z)
    rl = sp.return_loss_db(g) if z != 50 else float("inf")
    print(f"{z:4d} ohm: Gamma {g:+.3f}, RL {rl:6.2f} dB, VSWR {sp.vswr(g):5.2f}, mismatch loss {sp.mismatch_loss_db(g):.3f} dB")

  10 ohm: Gamma -0.667, RL   3.52 dB, VSWR  5.00, mismatch loss 2.553 dB
  25 ohm: Gamma -0.333, RL   9.54 dB, VSWR  2.00, mismatch loss 0.512 dB
  50 ohm: Gamma +0.000, RL    inf dB, VSWR  1.00, mismatch loss -0.000 dB
  75 ohm: Gamma +0.200, RL  13.98 dB, VSWR  1.50, mismatch loss 0.177 dB
 100 ohm: Gamma +0.333, RL   9.54 dB, VSWR  2.00, mismatch loss 0.512 dB
 250 ohm: Gamma +0.667, RL   3.52 dB, VSWR  5.00, mismatch loss 2.553 dB


## Part 2 · ngspice S-parameters

*Predict* the three networks' S-parameters at 1 GHz.

In [3]:
print("Python (a):", np.round(sp.s_series(10 + 40j)[:, 0], 4))
print("Python (c):", np.round(sp.s_line(75, math.pi / 4)[:, 0], 4))
for k, val in ngspice("sparams.cir").items():
    print(f"  {k} = {val:.4f}")

Python (a): [0.1971+0.292j 0.8029-0.292j]
Python (c): [0.2077+0.1917j 0.6506-0.7048j]
  as11m = 0.3523
  as11p = 55.9806
  as21m = 0.8544
  as21p = -19.9831
  bs21db = -6.0129
  bs11db = -67.0359
  cs11m = 0.2826
  cs11p = 42.7094
  cs21m = 0.9592
  cs21p = -47.2906


## Part 3 · Conversions and cascades

In [4]:
z = np.array([[120, 100], [100, 120]], dtype=complex)
s = sp.z_to_s(z)
print("S =", np.round(s, 4))
print("round trip Z:", np.round(sp.s_to_z(s), 6).real)
print("reciprocal:", np.isclose(s[0, 1], s[1, 0]), " power sum:", round(abs(s[0, 0]) ** 2 + abs(s[1, 0]) ** 2, 4))
k = 10 ** (3 / 20)
rsh, rse = 50 * (k + 1) / (k - 1), 50 * (k * k - 1) / (2 * k)
pad = sp.abcd_to_s(np.array([[1 + rse / rsh, rse], [2 / rsh + rse / rsh ** 2, 1 + rse / rsh]]))
tot = sp.t_to_s(sp.s_to_t(pad) @ sp.s_to_t(sp.s_line(50, math.pi / 3)) @ sp.s_to_t(pad))
print(f"cascade S21: {20 * np.log10(abs(tot[1, 0])):.2f} dB at {math.degrees(cmath.phase(tot[1, 0])):.1f} deg")

S = [[0.1005+0.j 0.5291+0.j]
 [0.5291+0.j 0.1005+0.j]]
round trip Z: [[120. 100.]
 [100. 120.]]
reciprocal: True  power sum: 0.2901
cascade S21: -6.00 dB at -60.0 deg


## Part 4 · Calibration

*Predict* the worst raw phase error.

In [5]:
e00, e11, t = 0.05 + 0.02j, 0.1 - 0.05j, P(0.9, -30)
meas = lambda g: e00 + t * g / (1 - e11 * g) + 0.0005 * (rng.normal() + 1j * rng.normal())
cal = sp.one_port_cal(meas(-1), meas(1), meas(0))
true = [P(0.5, a) for a in range(-180, 180, 15)]
raw = [meas(g) for g in true]
cor = [sp.one_port_correct(m, *cal) for m in raw]
print(f"worst raw error {max(abs(r - g) for r, g in zip(raw, true)):.3f}, after calibration {max(abs(c - g) for c, g in zip(cor, true)):.4f}")

worst raw error 0.285, after calibration 0.0019


## Part 5 · Stability

*Predict* where on the load plane the 500 MHz transistor becomes unstable.

In [6]:
s500 = np.array([[P(0.85, -60), P(0.03, 60)], [P(9, 120), P(0.7, -25)]])
x = np.linspace(-1, 1, 301)
gx, gy = np.meshgrid(x, x)
gl = gx + 1j * gy
gin = s500[0, 0] + s500[0, 1] * s500[1, 0] * gl / (1 - s500[1, 1] * gl)
mask = (np.abs(gl) < 1) & (np.abs(gin) > 1)
c, r = sp.stability_circle(s500)
th = np.linspace(0, 2 * np.pi, 400)
plt.figure(figsize=(5, 5))
plt.contourf(gx, gy, mask, levels=[0.5, 1.5], colors=["tab:red"], alpha=0.4)
plt.plot(np.cos(th), np.sin(th), "k")
plt.plot((c + r * np.exp(1j * th)).real, (c + r * np.exp(1j * th)).imag, "b--")
plt.xlim(-1.1, 1.1); plt.ylim(-1.1, 1.1); plt.gca().set_aspect("equal"); plt.title("unstable loads (red)"); plt.show()
print("stability:", {k: round(float(v), 3) for k, v in sp.stability(s500).items() if not np.isnan(v)})
s11p = np.array([[P(0.7, -120), P(0.08, 40)], [P(3.5, 90), P(0.45, -50)]])
for rsh in (100, 200, 240, 300, 500):
    st = sp.stability(sp.t_to_s(sp.s_to_t(s11p) @ sp.s_to_t(sp.s_shunt(1 / rsh))))
    print(f"shunt {rsh:4d} ohm: K {st['K']:.3f}, mu {st['mu']:.3f}")

stability: {'K': 0.449, 'delta': 0.674, 'mu': 0.498, 'MSG': 300.0}
shunt  100 ohm: K 1.409, mu 1.322
shunt  200 ohm: K 1.059, mu 1.046
shunt  240 ohm: K 1.000, mu 1.000
shunt  300 ohm: K 0.942, mu 0.956
shunt  500 ohm: K 0.849, mu 0.886
